# Cake or Dog — interactive notebook

This notebook is intended for interactive work with the Cake-or-Dog classifier.

Main goals:
- model training and evaluation
- error visualization
- interactive predictions

## 0. Setup

Make sure that Jupyter is running in the same venv as the project.

In [ ]:
# !pip install -e .

import sys
print(sys.executable)

In [ ]:
from pathlib import Path
import pandas as pd

from cakeordog.data import (
    load_split_parallel,
    load_single_image,
    CATEGORIES,
)
from cakeordog.model import (
    train_svm_gridsearch,
    save_model,
    load_model,
    predict_label,
)

## 1. Paths to data and model

In [ ]:
DATA_TRAIN = Path("../data/train")
DATA_TEST = Path("../data/test")
MODEL_OUT_DUMMY = Path("../models/notebook_model.joblib")
MODEL_OUT = Path("../models/svc_muffin_chihuahua.joblib")
ANTI_ALIASING=False

## 2. Loading test data

In [ ]:
x_test, y_test = load_split_parallel(DATA_TEST, anti_aliasing=ANTI_ALIASING)

print(f"Amount of processed test files: {len(x_test)}")

### 2.1 Show processed image

In [ ]:
from IPython.display import display
from PIL import Image
import numpy as np

x = x_test[0]
print(x)
x = x.reshape(32, 32, 3)
x = x.astype(np.uint8)
im = Image.fromarray(x)
im = im.convert('RGB')
im = im.resize((256, 256))

display(im)

## 3. Model training

SVM + GridSearch is used.
The result contains the best model and metrics on the test set.

***Warning***: For training you must download dataset from https://www.kaggle.com/datasets/samuelcortinhas/muffin-vs-chihuahua-image-classification/data

In [ ]:
# Amount of data for training.
# Use function load_split_parallel without that variable for taking all data
MAX_AMOUNT = 128

x_train, y_train = load_split_parallel(DATA_TRAIN, MAX_AMOUNT, anti_aliasing=ANTI_ALIASING)
print(f"Amount of processed train files: {len(x_train)}")

In [ ]:
result = train_svm_gridsearch(
    data_train=x_train,
    labels_train=y_train,
    data_test=x_test,
    labels_test=y_test,
)

result.best_params, result.test_accuracy

## 4. Saving the model

In [ ]:
MODEL_OUT_DUMMY.parent.mkdir(exist_ok=True)
save_model(result.best_model, MODEL_OUT_DUMMY)
MODEL_OUT_DUMMY

## 5. Batch predict

Run the model on the test dataset and collect the results into a DataFrame

### Our model

In [ ]:
model = load_model(MODEL_OUT_DUMMY)

rows = []
for img in DATA_TEST.rglob("*.jpg"):
    x = load_single_image(img, anti_aliasing=ANTI_ALIASING)
    pred = predict_label(model, x)
    rows.append({
        "path": str(img),
        "true": img.parent.name,
        "pred": CATEGORIES[pred],
    })

df = pd.DataFrame(rows)
df.head()

#### Accuracy and confusion matrix

In [ ]:
(df.true == df.pred).mean()

In [ ]:
pd.crosstab(df.true, df.pred, normalize="index")

### Pretrained model

In [ ]:
model = load_model(MODEL_OUT)

rows = []
for img in DATA_TEST.rglob("*.jpg"):
    x = load_single_image(img, anti_aliasing=True)
    pred = predict_label(model, x)
    rows.append({
        "path": str(img),
        "true": img.parent.name,
        "pred": CATEGORIES[pred],
    })

df = pd.DataFrame(rows)
df.head()

#### Accuracy and confusion matrix

In [ ]:
(df.true == df.pred).mean()

In [ ]:
pd.crosstab(df.true, df.pred, normalize="index")

## 6. Error inspection

Interactive slider for viewing incorrectly classified images

In [ ]:
from ipywidgets import interact
from PIL import Image

errors = df[df.true != df.pred].reset_index(drop=True)
len(errors)

In [ ]:
from IPython.display import display
from PIL import Image

MAX_HEIGHT = 300  # пиксели

@interact(i=(0, len(errors)-1))
def show_error(i):
    row = errors.iloc[i]
    img = Image.open(row.path)
    
    # сохраняем пропорции, ограничиваем высоту
    w, h = img.size
    new_w = int(w * MAX_HEIGHT / h)
    img = img.resize((new_w, MAX_HEIGHT))
    
    display(img)
    print("true:", row.true)
    print("pred:", row.pred)

## 7. Single image prediction

In [ ]:
from IPython.display import display

MAX_HEIGHT = 300


for path in [str(DATA_TEST) + "/chihuahua/img_0_580.jpg", str(DATA_TEST) + "/muffin/img_0_602.jpg"]:
    img_path = Path(path)
    img = Image.open(img_path)
    
    w, h = img.size
    if h > MAX_HEIGHT:
        new_w = int(w * MAX_HEIGHT / h)
        img = img.resize((new_w, MAX_HEIGHT))
    
    display(img)
    x = load_single_image(img_path)
    pred = predict_label(model, x)
    display(CATEGORIES[pred])

### 8. Our friends

Let us know what our friends are

Before running script put photos of your friends to data/friends folder

In [ ]:
from IPython.display import display
from pathlib import Path

MAX_HEIGHT = 450

model = load_model(MODEL_OUT)

folder_path = Path("../data/friends")
image_files = list(folder_path.iterdir())

result_count = {CATEGORIES[0]: 0, CATEGORIES[1]: 0}
results = []

for path in image_files:
    img = Image.open(path)

    w, h = img.size
    if h > MAX_HEIGHT:
        new_w = int(w * MAX_HEIGHT / h)
        img = img.resize((new_w, MAX_HEIGHT))


    x = load_single_image(path, anti_aliasing=False)
    pred = predict_label(model, x)
    results.append((img, CATEGORIES[pred]))
    result_count[CATEGORIES[pred]] += 1

print(result_count)

for img, pred in results:
    display(img)
    display(pred)